<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session01_getting_started_with_llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱: شروع کار با مدل‌های زبانی بزرگ

### اهداف این جلسه

- دیدن LLM از درون: پیش‌بینی توکن بعدی، توزیع احتمال و حلقه تولید متن
- آشنایی با سه مرحله ساخت یک دستیار: Pretraining، Instruction Tuning و هم‌راستاسازی
- درک توکن‌سازی و اینکه چرا متن فارسی توکن بیشتری مصرف می‌کند
- اجرای یک مدل متن‌باز با Ollama و فراخوانی آن با API سازگار با OpenAI
- کار با نقش‌های گفتگو، پارامترهای تولید، استریم پاسخ و برآورد هزینه

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. مدل زبانی چه کاری انجام می‌دهد؟

در جلسه ۱۳ دوره یادگیری عمیق یک GPT کوچک را از صفر ساختیم. مدل‌های زبانی بزرگ (Large Language Models) همان معماری Transformer را دارند، با سه تفاوت: میلیاردها پارامتر، آموزش روی تریلیون‌ها توکن متن و چند مرحله آموزش اضافه که آن‌ها را به دستیار تبدیل می‌کند.

کار اصلی هر مدل زبانی یک چیز است: **پیش‌بینی توکن بعدی (Next-Token Prediction)**. مدل دنباله‌ای از توکن‌ها را می‌گیرد و برای همه توکن‌های واژگان یک **توزیع احتمال** برمی‌گرداند:

$$P(x_{t} \mid x_{1}, x_{2}, \dots, x_{t-1})$$

تولید متن یعنی تکرار این کار: پیش‌بینی توزیع، انتخاب یک توکن، افزودن آن به ورودی و دوباره پیش‌بینی. به این روش **تولید خودبازگشتی (Autoregressive Generation)** می‌گوییم.

برای دیدن این فرایند از درون، از یک مدل کوچک با ۰٫۵ میلیارد پارامتر استفاده می‌کنیم که با کتابخانه `transformers` روی همین ماشین اجرا می‌شود.

</div>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME, torch_dtype="auto", device_map="auto")

print(f"parameters: {lm.num_parameters() / 1e6:.0f}M   vocabulary: {len(tokenizer):,} tokens")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تابع زیر یک متن می‌گیرد و ده توکن محتمل بعدی را همراه با احتمال آن‌ها نشان می‌دهد. خروجی لایه آخر مدل **لاجیت (Logit)** است و با Softmax به احتمال تبدیل می‌شود.

</div>

In [ ]:
def next_token_probs(text, k=10):
    """Return the k most likely next tokens and their probabilities."""
    inputs = tokenizer(text, return_tensors="pt").to(lm.device)
    with torch.no_grad():
        logits = lm(**inputs).logits[0, -1]          # logits of the last position
    probs = torch.softmax(logits.float(), dim=-1)
    top = torch.topk(probs, k)
    return [(tokenizer.decode([int(i)]), p.item()) for p, i in zip(top.values, top.indices)]


for text in ["The capital of France is", "پایتخت ایران شهر"]:
    print(f"\n{text!r}")
    for token, p in next_token_probs(text):
        print(f"  {p:6.3f}  {token!r}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

حالا حلقه تولید متن را خودمان می‌نویسیم. در هر گام محتمل‌ترین توکن را انتخاب می‌کنیم. به این روش **رمزگشایی حریصانه (Greedy Decoding)** می‌گوییم. تابع `generate` در کتابخانه‌ها همین حلقه را با جزئیات بیشتر اجرا می‌کند.

</div>

In [ ]:
def greedy_generate(text, n_tokens=20):
    ids = tokenizer(text, return_tensors="pt").input_ids.to(lm.device)
    for _ in range(n_tokens):
        with torch.no_grad():
            next_id = lm(ids).logits[0, -1].argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
        print(repr(tokenizer.decode([int(next_id)])), end=" ")
    return tokenizer.decode(ids[0])


print("\n\n" + greedy_generate("Machine learning is"))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. دما (Temperature) و نمونه‌برداری

انتخاب همیشگی محتمل‌ترین توکن خروجی قطعی اما تکراری می‌دهد. در عمل از توزیع **نمونه‌برداری (Sampling)** می‌کنیم. پیش از Softmax، لاجیت‌ها را بر **دما** $T$ تقسیم می‌کنیم:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- $T < 1$ توزیع را تیزتر می‌کند و خروجی قابل پیش‌بینی‌تر می‌شود.
- $T > 1$ توزیع را پهن‌تر می‌کند و خروجی متنوع‌تر و پرخطاتر می‌شود.
- $T = 0$ معادل همان رمزگشایی حریصانه است.

نمودار زیر اثر دما را روی توزیع توکن بعدی نشان می‌دهد:

</div>

In [ ]:
import matplotlib.pyplot as plt

inputs = tokenizer("My favourite food is", return_tensors="pt").to(lm.device)
with torch.no_grad():
    logits = lm(**inputs).logits[0, -1].float()
top_ids = torch.topk(logits, 8).indices
labels = [tokenizer.decode([int(i)]).strip() for i in top_ids]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, T in zip(axes, [0.3, 1.0, 2.0]):
    probs = torch.softmax(logits / T, dim=-1)[top_ids].cpu()
    ax.bar(labels, probs)
    ax.set_title(f"T = {T}")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("probability")
plt.tight_layout()
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**پارامتر top_p** راه دیگری برای کنترل نمونه‌برداری است: فقط از کوچک‌ترین مجموعه توکن‌هایی نمونه می‌گیریم که مجموع احتمالشان به $p$ برسد (**Nucleus Sampling**). معمولاً یکی از این دو پارامتر را تنظیم می‌کنیم، نه هر دو را.

## ۳. توکن‌سازی (Tokenization)

مدل متن را به صورت **توکن** می‌بیند، نه حرف یا کلمه. توکن‌ساز این مدل از **BPE** (Byte-Pair Encoding) استفاده می‌کند: واحدهای پرتکرار متن آموزشی به یک توکن تبدیل شده‌اند. چون بیشتر داده آموزشی انگلیسی بوده، کلمات انگلیسی معمولاً یک توکن‌اند، ولی کلمات فارسی به چند تکه شکسته می‌شوند.

</div>

In [ ]:
sentences = [
    "The weather is very nice today.",
    "امروز هوا خیلی خوب است.",
]
for s in sentences:
    ids = tokenizer(s).input_ids
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{len(s):3d} characters -> {len(ids):3d} tokens   {pieces}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این تفاوت سه پیامد عملی دارد:

1. **هزینه:** سرویس‌های API به ازای توکن هزینه می‌گیرند، پس یک متن فارسی معمولاً گران‌تر از معادل انگلیسی آن است.
2. **پنجره متن (Context Window):** سقف ورودی مدل بر حسب توکن است، پس متن فارسی زودتر به سقف می‌رسد.
3. **سرعت:** تولید هر توکن زمان می‌برد، پس پاسخ فارسی کندتر تولید می‌شود.

## ۴. از تکمیل‌کننده متن تا دستیار

مدلی که فقط برای پیش‌بینی توکن بعدی آموزش دیده (**Base Model**) متن را ادامه می‌دهد، ولی دستور را اجرا نمی‌کند. دستیارهایی مثل ChatGPT سه مرحله آموزش را طی کرده‌اند:

| مرحله | داده | نتیجه |
|---|---|---|
| Pretraining | تریلیون‌ها توکن متن وب، کتاب و کد | دانش زبان و دنیا؛ ادامه دادن متن |
| Instruction Tuning (SFT) | ده‌ها هزار جفت دستور و پاسخ خوب | پیروی از دستور و قالب گفتگو |
| هم‌راستاسازی (RLHF یا DPO) | مقایسه‌های انسانی بین پاسخ‌ها | پاسخ‌های مفیدتر، صادق‌تر و ایمن‌تر |

مدل‌هایی که با پسوند `Instruct` یا `Chat` منتشر می‌شوند از هر سه مرحله گذشته‌اند.

### گفتگو هم در نهایت متن است

مدل دستیار باز هم فقط توکن بعدی را پیش‌بینی می‌کند. گفتگو با یک **قالب گفتگو (Chat Template)** به متن تبدیل می‌شود: هر پیام بین توکن‌های ویژه قرار می‌گیرد و نقش گوینده مشخص می‌شود.

</div>

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "سلام! حالت چطوره؟"},
]
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل با دیدن `<|im_start|>assistant` می‌فهمد که نوبت پاسخ دادن است و تولید را تا رسیدن به توکن پایان `<|im_end|>` ادامه می‌دهد. هر خانواده مدل قالب خودش را دارد و اگر قالب اشتباه باشد، کیفیت پاسخ به‌شدت افت می‌کند. این نکته در جلسه ۱۰، هنگام فاین‌تیون، دوباره مهم می‌شود.

کار ما با مدل کوچک تمام شد. حافظه GPU را آزاد می‌کنیم تا برای مدل بزرگ‌تر جا باشد:

</div>

In [ ]:
del lm
torch.cuda.empty_cache()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. اجرای مدل متن‌باز با Ollama

برای ساخت برنامه، به جای کار مستقیم با `transformers` از یک **سرور مدل (Model Server)** استفاده می‌کنیم. [Ollama](https://ollama.com) مدل‌های متن‌باز را با کوانتیزه‌سازی ۴ بیتی اجرا می‌کند. به همین دلیل مدل ۷ میلیارد پارامتری روی GPU رایگان کولب، و حتی روی لپ‌تاپ، جا می‌شود. Ollama یک API سازگار با OpenAI هم ارائه می‌دهد.

تابع `setup_llm` در فایل `llm_utils.py` این کارها را انجام می‌دهد:

1. اگر Ollama نصب نیست، با اسکریپت رسمی نصبش می‌کند.
2. سرور را با دستور `ollama serve` در پس‌زمینه اجرا می‌کند. پنجره متن روی ۸۱۹۲ توکن تنظیم می‌شود، چون مقدار پیش‌فرض Ollama کوچک است و ورودی‌های بلندتر **بی‌صدا بریده می‌شوند**.
3. مدل را با `ollama pull` دانلود می‌کند. مدل پیش‌فرض `qwen2.5:7b` است که فارسی را نسبتاً خوب می‌فهمد و از فراخوانی ابزار هم پشتیبانی می‌کند.
4. یک کلاینت `OpenAI` می‌سازد که به جای سرورهای OpenAI به `http://localhost:11434/v1` وصل می‌شود.

اگر به یک سرویس API دسترسی دارید، `setup_llm("api")` را اجرا کنید. بقیه کد نوت‌بوک هیچ تغییری نمی‌کند.

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. اولین فراخوانی

درخواست به **Chat Completions API** شامل نام مدل و فهرست پیام‌هاست. هر پیام یک **نقش (Role)** دارد:

- `system`: دستورالعمل کلی رفتار مدل. کاربر نهایی معمولاً آن را نمی‌بیند.
- `user`: پیام کاربر.
- `assistant`: پاسخ‌های قبلی مدل.

</div>

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "تو یک دستیار فارسی‌زبان هستی و کوتاه و دقیق پاسخ می‌دهی."},
        {"role": "user", "content": "در دو جمله توضیح بده یادگیری ماشین چیست."},
    ],
)

print(response.choices[0].message.content)
print("\nfinish_reason:", response.choices[0].finish_reason)
print("usage:", response.usage)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در پاسخ، علاوه بر متن، دو فیلد مهم هست:

- `finish_reason`: مقدار `stop` یعنی مدل خودش پاسخ را تمام کرده و `length` یعنی به سقف `max_tokens` رسیده و پاسخ ناقص است.
- `usage`: تعداد توکن‌های ورودی (`prompt_tokens`) و خروجی (`completion_tokens`). هزینه API بر همین اساس محاسبه می‌شود.

## ۷. مدل حافظه ندارد

API **بی‌حالت (Stateless)** است: مدل هیچ چیزی از درخواست‌های قبلی به خاطر نمی‌آورد. برای ادامه گفتگو باید کل تاریخچه را در هر درخواست دوباره بفرستیم.

</div>

In [ ]:
history = [{"role": "system", "content": "تو یک دستیار فارسی‌زبان هستی و کوتاه پاسخ می‌دهی."}]


def ask(question):
    history.append({"role": "user", "content": question})
    answer = chat(client, MODEL, messages=history)
    history.append({"role": "assistant", "content": answer})
    return answer


print(ask("اسم من کیوان است و برنامه‌نویس پایتون هستم."))
print("---")
print(ask("اسم من چه بود و چه کاری انجام می‌دهم؟"))
print("---")
print("without history:", chat(client, MODEL, "اسم من چه بود؟"))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

چون کل تاریخچه هر بار ارسال می‌شود، گفتگوهای طولانی به‌تدریج گران و کند می‌شوند و در نهایت به سقف پنجره متن می‌رسند. مدیریت حافظه گفتگو موضوع جلسه ۱۵ است.

## ۸. پارامترهای تولید

| پارامتر | کاربرد | مقدار رایج |
|---|---|---|
| `temperature` | میزان تصادفی بودن نمونه‌برداری | ۰ برای استخراج و طبقه‌بندی، ۰٫۷ تا ۱ برای تولید خلاقانه |
| `top_p` | نمونه‌برداری از هسته احتمال | ۰٫۹ تا ۱ |
| `max_tokens` | سقف طول پاسخ | بسته به کاربرد |
| `seed` | تکرارپذیری نسبی نتایج | یک عدد ثابت |
| `stop` | رشته‌هایی که تولید را متوقف می‌کنند | مثلاً `["\n\n"]` |

آزمایش زیر یک درخواست خلاقانه را پنج بار با دو دمای مختلف اجرا می‌کند:

</div>

In [ ]:
prompt = "یک نام کوتاه و خلاقانه برای یک کافه‌کتاب پیشنهاد بده. فقط نام را بنویس."

for temperature in [0.0, 1.2]:
    names = [chat(client, MODEL, prompt, temperature=temperature, max_tokens=20).strip()
             for _ in range(5)]
    print(f"temperature={temperature}: {names}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با دمای صفر هر پنج پاسخ یکسان است و با دمای بالا پاسخ‌ها متنوع می‌شوند. حالا اثر `max_tokens` را ببینیم:

</div>

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "تاریخچه زبان پایتون را توضیح بده."}],
    max_tokens=30,
)
print(response.choices[0].message.content)
print("\nfinish_reason:", response.choices[0].finish_reason)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در برنامه واقعی همیشه `finish_reason` را بررسی کنید. پاسخ بریده‌شده، به‌خصوص وقتی خروجی باید JSON باشد، باعث خطا در ادامه برنامه می‌شود.

## ۹. استریم پاسخ (Streaming)

در حالت عادی کل پاسخ پس از تولید کامل برگردانده می‌شود. با `stream=True` توکن‌ها به محض تولید ارسال می‌شوند. چت‌بات‌ها همین‌طور کار می‌کنند و کاربر به جای چند ثانیه انتظار، بلافاصله شروع پاسخ را می‌بیند.

</div>

In [ ]:
import time

start = time.time()
first_token_at = None
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "سه مزیت یادگیری زبان پایتون را بنویس."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first_token_at is None:
            first_token_at = time.time() - start
        print(delta, end="", flush=True)

print(f"\n\ntime to first token: {first_token_at:.2f}s   total: {time.time() - start:.2f}s")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱۰. سرعت و هزینه

دو معیار اصلی برای مقایسه مدل‌ها و سرویس‌ها: **سرعت تولید** بر حسب توکن در ثانیه و **هزینه** بر حسب قیمت هر میلیون توکن ورودی و خروجی. تابع زیر هر دو را برای یک درخواست اندازه می‌گیرد.

</div>

In [ ]:
def measure(prompt, price_in=0.15, price_out=0.60, **kwargs):
    """Run one request and report speed and cost.

    Prices are in dollars per million tokens. The defaults are only an example;
    check the price list of the service you use.
    """
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], **kwargs)
    elapsed = time.time() - start
    usage = response.usage
    cost = (usage.prompt_tokens * price_in + usage.completion_tokens * price_out) / 1e6
    print(f"input={usage.prompt_tokens}  output={usage.completion_tokens}  "
          f"time={elapsed:.1f}s  speed={usage.completion_tokens / elapsed:.1f} tok/s  "
          f"cost=${cost:.6f}")


measure("Explain overfitting in three sentences.")
measure("بیش‌برازش (Overfitting) را در سه جمله توضیح بده.")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱۱. مدل محلی یا API؟

| معیار | مدل محلی (Ollama، vLLM) | API تجاری (OpenAI و ...) | درگاه‌های داخلی سازگار با OpenAI |
|---|---|---|---|
| کیفیت | مدل‌های کوچک ضعیف‌تر از مدل‌های بزرگ تجاری‌اند | بهترین کیفیت فعلی | همان مدل‌های تجاری |
| حریم خصوصی | داده از سازمان خارج نمی‌شود | داده به سرور خارجی می‌رود | داده به ارائه‌دهنده می‌رود |
| هزینه | هزینه ثابت سخت‌افزار | به ازای توکن، ارزی | به ازای توکن، ریالی |
| دسترسی | همیشه در دسترس | وابسته به تحریم و فیلترینگ | وابسته به ارائه‌دهنده |

برای ادامه دوره کد ما به این انتخاب وابسته نیست، چون همه سرویس‌ها از یک API پیروی می‌کنند. یک الگوی رایج در پروژه‌های واقعی این است: نمونه اولیه با بهترین مدل API ساخته می‌شود و بعد بررسی می‌شود آیا یک مدل محلی کوچک‌تر، یا فاین‌تیون‌شده، به کیفیت کافی می‌رسد.

## ۱۲. جمع‌بندی

- LLM در هر گام توزیع احتمال توکن بعدی را پیش‌بینی می‌کند و متن با تکرار این گام تولید می‌شود.
- دما و top_p تعیین می‌کنند از این توزیع چگونه نمونه بگیریم.
- دستیارها حاصل Pretraining، Instruction Tuning و هم‌راستاسازی‌اند و گفتگو با یک قالب ویژه به متن تبدیل می‌شود.
- متن فارسی توکن بیشتری مصرف می‌کند و این روی هزینه، سرعت و پنجره متن اثر دارد.
- API سازگار با OpenAI کد ما را از سرویس‌دهنده مستقل می‌کند و مدل حافظه ندارد، پس تاریخچه را خودمان می‌فرستیم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. مدل `gemma3:4b` یا `llama3.2:3b` را با `setup_llm("ollama", model="...")` اجرا کنید. پاسخ آن به سه پرسش فارسی را با `qwen2.5:7b` مقایسه کنید و سرعت هر دو را با تابع `measure` بسنجید.
۲. یک پاراگراف فارسی و ترجمه انگلیسی آن را انتخاب کنید و تعداد توکن‌هایشان را با توکن‌ساز Qwen بشمارید. نسبت توکن‌ها چقدر است؟ این نسبت چه اثری روی هزینه یک چت‌بات فارسی دارد؟
۳. تابع `ask` را طوری تغییر دهید که فقط ۶ پیام آخر تاریخچه، به علاوه پیام system، ارسال شود. گفتگویی طراحی کنید که نشان دهد مدل چه چیزی را فراموش می‌کند.
۴. با `temperature=0.7` و یک `seed` ثابت، یک درخواست را سه بار اجرا کنید. آیا خروجی‌ها یکسان است؟ بدون `seed` چطور؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** مدل `gemma3:4b` یا `llama3.2:3b` را با `setup_llm("ollama", model="...")` اجرا کنید. پاسخ آن به سه پرسش فارسی را با `qwen2.5:7b` مقایسه کنید و سرعت هر دو را با تابع `measure` بسنجید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** یک پاراگراف فارسی و ترجمه انگلیسی آن را انتخاب کنید و تعداد توکن‌هایشان را با توکن‌ساز Qwen بشمارید. نسبت توکن‌ها چقدر است؟ این نسبت چه اثری روی هزینه یک چت‌بات فارسی دارد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** تابع `ask` را طوری تغییر دهید که فقط ۶ پیام آخر تاریخچه، به علاوه پیام system، ارسال شود. گفتگویی طراحی کنید که نشان دهد مدل چه چیزی را فراموش می‌کند.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** با `temperature=0.7` و یک `seed` ثابت، یک درخواست را سه بار اجرا کنید. آیا خروجی‌ها یکسان است؟ بدون `seed` چطور؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید